# Waste Classification Model Fine-Tuning

This notebook fine-tunes the YOLOv8 waste classification model using user feedback data collected from the app.
It is designed to run fully automatically — triggered by the `retrain-orchestrator` Cloud Function every 3 days.

## Data Source & Organization
Training data is stored in **Google Cloud Storage (GCS)**:

| Folder | Purpose |
|--------|---------|
| `training_data/images/` | **Pending** - New photos to be trained on |
| `training_data/labels/` | **Pending** - Labels for new photos |
| `trained_data/{timestamp}/images/` | **Completed** - Already trained photos |
| `trained_data/{timestamp}/labels/` | **Completed** - Already trained labels |
| `models/best_latest.pt` | Current production weights (downloaded + re-uploaded here) |

After training completes, files are **always** moved from `training_data/` to `trained_data/{timestamp}/`
regardless of whether the new model improved. This prevents the scheduler from re-running on the same data.

## Prerequisites (one-time setup only)
1. Upload your `serviceAccountKey.json` as a Kaggle Secret named `FIREBASE_CREDENTIALS`
2. Upload your initial `best.pt` to GCS **once**:
   ```
   gsutil cp best.pt gs://retrain_smart_waste_model/models/best_latest.pt
   ```
   Every subsequent run downloads the latest deployed weights from GCS automatically — no manual upload needed.
3. In the **Session options** panel (right side) → **Accelerator** → select **GPU T4 x2**, then save the notebook. This setting persists for all future automated runs.

## Training Strategy
- **Fine-tuning** (not training from scratch) — preserves learned features
- Downloads current production weights from GCS as the starting point
- Lower learning rate (0.001) to avoid catastrophic forgetting
- Freezes early backbone layers to maintain general feature extraction
- Early stopping to prevent overfitting
- Validates both the old and new model on the same dataset; only deploys if new mAP50 > baseline mAP50

## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q firebase-admin ultralytics pillow google-cloud-storage

In [ ]:
import os
import json
import random
from pathlib import Path
from datetime import datetime

import firebase_admin
from firebase_admin import credentials, storage, firestore
from ultralytics import YOLO

# Kaggle paths
WORKING_DIR = Path("/kaggle/working")
DATASET_DIR = WORKING_DIR / "feedback_dataset"
OUTPUT_DIR = WORKING_DIR / "training_output"

# Configuration
BUCKET_NAME = "retrain_smart_waste_model"
MIN_SAMPLES_REQUIRED = 1000  # Minimum feedback samples to trigger training

print(f"Working directory: {WORKING_DIR}")
print(f"Dataset will be saved to: {DATASET_DIR}")

## 2. Initialize GCS Access

The Firebase Admin SDK provides access to Google Cloud Storage buckets.
Load credentials from Kaggle Secrets to authenticate.

In [ ]:
from kaggle_secrets import UserSecretsClient

# Get Firebase credentials from Kaggle Secrets
secrets = UserSecretsClient()
firebase_creds_json = secrets.get_secret("FIREBASE_CREDENTIALS")
firebase_creds = json.loads(firebase_creds_json)

# Initialize Firebase
if not firebase_admin._apps:
    cred = credentials.Certificate(firebase_creds)
    firebase_admin.initialize_app(cred)

db = firestore.client()
bucket = storage.bucket(BUCKET_NAME)

print("Firebase initialized successfully!")

## 3. Check Available Training Data in GCS

List files directly from the GCS bucket to see how many labeled samples are available.

In [ ]:
def get_gcs_training_data_count():
    """
    Count training samples by listing files directly from GCS bucket.
    A valid sample = image + corresponding label file.
    """
    label_blobs = list(bucket.list_blobs(prefix="training_data/labels/"))
    label_files = [b.name for b in label_blobs if b.name.endswith('.txt')]

    image_blobs = list(bucket.list_blobs(prefix="training_data/images/"))
    image_files = {b.name.replace("training_data/images/", "").replace(".jpg", "")
                   for b in image_blobs if b.name.endswith('.jpg')}

    valid_samples = sum(
        1 for lp in label_files
        if lp.replace("training_data/labels/", "").replace(".txt", "") in image_files
    )

    return {
        "total_images": len(image_files),
        "total_labels": len(label_files),
        "valid_pairs": valid_samples
    }

# Check what's in the GCS bucket
print("Scanning GCS bucket for training data...")
gcs_stats = get_gcs_training_data_count()

print(f"\n{'='*50}")
print("GCS BUCKET CONTENTS")
print(f"{'='*50}")
print(f"Images in training_data/images/: {gcs_stats['total_images']}")
print(f"Labels in training_data/labels/: {gcs_stats['total_labels']}")
print(f"Valid image+label pairs:         {gcs_stats['valid_pairs']}")
print(f"{'='*50}")

feedback_count = gcs_stats['valid_pairs']
SHOULD_TRAIN = feedback_count >= MIN_SAMPLES_REQUIRED

if not SHOULD_TRAIN:
    print(f"\nNot enough training data: {feedback_count}/{MIN_SAMPLES_REQUIRED} valid pairs.")
    print("Writing skip status to GCS and stopping gracefully.")
    bucket.blob("models/training_status.json").upload_from_string(
        json.dumps({
            "status": "skipped",
            "reason": f"Not enough training data: {feedback_count}/{MIN_SAMPLES_REQUIRED} valid pairs",
            "completed_at": datetime.utcnow().isoformat() + "Z",
            "samples_available": feedback_count,
            "improved": False,
        }),
        content_type="application/json",
    )
    print("Skip status written to GCS: models/training_status.json")
else:
    print(f"\nSufficient samples available ({feedback_count}). Proceeding with training.")

## 4. Download Training Data from GCS

Downloads images and labels from `gs://retrain_smart_waste_model/training_data/`

In [ ]:
# Initialize tracking variables — defined here so archive cell always has them
downloaded_image_ids = []
total_downloaded = 0

def download_training_data(max_samples=None):
    """
    Download images and labels directly from GCS bucket.
    Lists files in the bucket and downloads matching image+label pairs.
    Also tracks which files were downloaded for later archiving.
    """
    global downloaded_image_ids
    downloaded_image_ids = []
    
    # Create directories
    images_dir = DATASET_DIR / "images" / "train"
    labels_dir = DATASET_DIR / "labels" / "train"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    # List all label files from GCS (these are our ground truth)
    print("Listing label files from GCS...")
    label_blobs = list(bucket.list_blobs(prefix="training_data/labels/"))
    label_files = [(b.name, b) for b in label_blobs if b.name.endswith('.txt')]
    print(f"Found {len(label_files)} label files")
    
    # Build set of available images for quick lookup
    print("Listing image files from GCS...")
    image_blobs = {b.name: b for b in bucket.list_blobs(prefix="training_data/images/") 
                   if b.name.endswith('.jpg')}
    print(f"Found {len(image_blobs)} image files")
    
    downloaded = 0
    skipped = 0
    
    for label_path, label_blob in label_files:
        if max_samples and downloaded >= max_samples:
            break
        
        # Extract image_id: training_data/labels/{uuid}.txt -> {uuid}
        image_id = label_path.replace("training_data/labels/", "").replace(".txt", "")
        image_path = f"training_data/images/{image_id}.jpg"
        
        # Check if corresponding image exists
        if image_path not in image_blobs:
            skipped += 1
            continue
        
        try:
            # Download image
            local_image = images_dir / f"{image_id}.jpg"
            image_blobs[image_path].download_to_filename(str(local_image))
            
            # Download label
            local_label = labels_dir / f"{image_id}.txt"
            label_blob.download_to_filename(str(local_label))
            
            # Track this file for archiving later
            downloaded_image_ids.append(image_id)
            
            downloaded += 1
            
            if downloaded % 50 == 0:
                print(f"Downloaded {downloaded} samples from GCS...")
                
        except Exception as e:
            print(f"Error downloading {image_id}: {e}")
            skipped += 1
    
    print(f"\nDownload complete: {downloaded} samples, {skipped} skipped")
    return downloaded

if SHOULD_TRAIN:
    total_downloaded = download_training_data()
    print(f"\nTracking {len(downloaded_image_ids)} files for archiving after training")
else:
    print("Skipping download (not enough samples).")

## 5. Create Train/Val Split

In [ ]:
def create_val_split(val_ratio=0.15):
    """Split data into training and validation sets."""
    
    images_train = DATASET_DIR / "images" / "train"
    labels_train = DATASET_DIR / "labels" / "train"
    images_val = DATASET_DIR / "images" / "val"
    labels_val = DATASET_DIR / "labels" / "val"
    
    images_val.mkdir(parents=True, exist_ok=True)
    labels_val.mkdir(parents=True, exist_ok=True)
    
    # Get all images
    image_files = list(images_train.glob("*.jpg"))
    random.shuffle(image_files)
    
    # Split
    val_count = int(len(image_files) * val_ratio)
    val_files = image_files[:val_count]
    
    print(f"Train: {len(image_files) - val_count}, Val: {val_count}")
    
    # Move validation files
    for img_path in val_files:
        image_id = img_path.stem
        
        # Move image
        img_path.rename(images_val / img_path.name)
        
        # Move label
        label_path = labels_train / f"{image_id}.txt"
        if label_path.exists():
            label_path.rename(labels_val / f"{image_id}.txt")
            
    print("Split complete!")

create_val_split(val_ratio=0.15)

## 6. Create Dataset Configuration

In [ ]:
# Create dataset.yaml for YOLO
dataset_yaml_content = f"""
# Waste Classification Fine-tuning Dataset
path: {DATASET_DIR}
train: images/train
val: images/val

# Classes
nc: 6
names:
  0: glass
  1: paper
  2: cardboard
  3: plastic
  4: metal
  5: trash
"""

dataset_yaml_path = DATASET_DIR / "dataset.yaml"
with open(dataset_yaml_path, 'w') as f:
    f.write(dataset_yaml_content)

print(f"Dataset config saved to: {dataset_yaml_path}")
print(dataset_yaml_content)

## 7. Load Base Model & Establish Baseline

Downloads the current production weights from GCS (`models/best_latest.pt`) automatically.
The model is then validated on the feedback dataset to record the **baseline mAP50** —
the quality bar the new model must beat before it gets deployed.

In [ ]:
if SHOULD_TRAIN:
    BASE_WEIGHTS_PATH = WORKING_DIR / "base_model.pt"

    print("Downloading base weights from GCS: gs://" + BUCKET_NAME + "/models/best_latest.pt")
    weights_blob = bucket.blob("models/best_latest.pt")
    if not weights_blob.exists():
        raise FileNotFoundError(
            "models/best_latest.pt not found in GCS. "
            "First-time setup: upload your best.pt to gs://" + BUCKET_NAME + "/models/best_latest.pt"
        )
    weights_blob.download_to_filename(str(BASE_WEIGHTS_PATH))
    print(f"Downloaded {BASE_WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB")

    model = YOLO(str(BASE_WEIGHTS_PATH))
    print(f"Loaded base model: {BASE_WEIGHTS_PATH}")

    print("\nValidating current model to establish baseline...")
    baseline_metrics = model.val(data=str(dataset_yaml_path))
    baseline_map50 = float(baseline_metrics.box.map50)
    print(f"\nBaseline mAP50 (current production model): {baseline_map50:.4f}")
else:
    print("Skipping base model download (not enough samples).")

## 8. Fine-tune the Model

Key hyperparameters for fine-tuning:
- `lr0=0.001`: Lower learning rate to preserve learned features
- `freeze=10`: Freeze first 10 layers (backbone) to avoid catastrophic forgetting
- `patience=15`: Early stopping if validation loss stops improving
- `device="auto"`: Automatically uses all available GPUs — on T4 x2 both GPUs train in parallel

In [ ]:
if SHOULD_TRAIN:
    EPOCHS = 50
    BATCH_SIZE = 16
    IMG_SIZE = 640
    LEARNING_RATE = 0.001
    FREEZE_LAYERS = 10

    print(f"\n{'='*60}")
    print("STARTING FINE-TUNING")
    print(f"{'='*60}")
    print(f"Epochs: {EPOCHS}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Freeze layers: {FREEZE_LAYERS}")
    print(f"{'='*60}\n")

    results = model.train(
        data=str(dataset_yaml_path),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        lr0=LEARNING_RATE,
        lrf=0.01,
        freeze=FREEZE_LAYERS,
        patience=15,
        save=True,
        save_period=10,
        project=str(OUTPUT_DIR),
        name="fine_tune",
        exist_ok=True,
        pretrained=True,
        optimizer="AdamW",
        weight_decay=0.0005,
        warmup_epochs=3,
        device="auto",
        verbose=True,
    )
else:
    print("Skipping training (not enough samples).")

## 9. Evaluate Results

In [ ]:
if SHOULD_TRAIN:
    best_model_path = OUTPUT_DIR / "fine_tune" / "weights" / "best.pt"

    if best_model_path.exists():
        print(f"Best model saved at: {best_model_path}")
        fine_tuned_model = YOLO(str(best_model_path))
        metrics = fine_tuned_model.val(data=str(dataset_yaml_path))

        print(f"\n{'='*60}")
        print("VALIDATION RESULTS")
        print(f"{'='*60}")
        print(f"mAP50: {metrics.box.map50:.4f}")
        print(f"mAP50-95: {metrics.box.map:.4f}")
        print(f"{'='*60}")
    else:
        print("Training may not have completed successfully.")
else:
    print("Skipping evaluation (not enough samples).")

## 10. Upload New Weights to GCS

Uploads the new `best.pt` to two GCS locations:
- `models/fine_tuned/best_{timestamp}.pt` — versioned copy for history and rollback
- `models/best_latest.pt` — the pointer Cloud Build uses when redeploying Cloud Run

Weights are always uploaded regardless of whether the model improved,
so the versioned history is always complete.
The Cloud Function only triggers a Cloud Run redeploy if `improved: true` in the status marker (written in Cell 11).

In [ ]:
if SHOULD_TRAIN and best_model_path.exists():
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    # Versioned copy (history / rollback)
    versioned_path = f"models/fine_tuned/best_{timestamp}.pt"
    bucket.blob(versioned_path).upload_from_filename(str(best_model_path))
    print(f"Uploaded versioned weights: gs://{BUCKET_NAME}/{versioned_path}")
else:
    print("Skipping weight upload (not enough samples or training did not complete).")

## 11. Training Summary

In [ ]:
import tempfile

if SHOULD_TRAIN:
    new_map50 = float(metrics.box.map50) if best_model_path.exists() else 0.0
    should_deploy = new_map50 > baseline_map50

    if should_deploy:
        bucket.blob("models/best_latest.pt").upload_from_filename(str(best_model_path))
        print(f"Updated latest: gs://{BUCKET_NAME}/models/best_latest.pt")

    print(f"\n{'='*60}")
    print("MODEL COMPARISON")
    print(f"{'='*60}")
    print(f"Baseline mAP50 (old model): {baseline_map50:.4f}")
    print(f"New model mAP50:            {new_map50:.4f}")
    print(f"Improvement:                {new_map50 - baseline_map50:+.4f}")
    print(f"Deploy decision:            {'YES - model improved' if should_deploy else 'NO - did not improve, keeping old model'}")
    print(f"{'='*60}")

    summary = {
        "completed_at": datetime.utcnow().isoformat(),
        "feedback_samples_used": total_downloaded,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "freeze_layers": FREEZE_LAYERS,
        "baseline_map50": baseline_map50,
        "new_map50": new_map50,
        "improved": should_deploy,
    }

    summary_path = OUTPUT_DIR / "training_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(json.dumps(summary, indent=2))

    status_marker = {
        "status": "complete",
        "completed_at": summary["completed_at"],
        "baseline_map50": baseline_map50,
        "new_map50": new_map50,
        "improved": should_deploy,
        "weights_path": "models/best_latest.pt",
        "samples_used": total_downloaded,
    }
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(status_marker, f)
        tmp_path = f.name
    bucket.blob("models/training_status.json").upload_from_filename(tmp_path)
    print("\nCompletion marker written to GCS: models/training_status.json")
else:
    print("Skipping training summary (not enough samples — skip status already written in cell 3).")

## 12. Archive Trained Data

Move the training data from `training_data/` to `trained_data/{timestamp}/` so you know what's been used.

In [ ]:
def archive_trained_data(image_ids, training_timestamp=None):
    """
    Move trained files from training_data/ to trained_data/{timestamp}/.
    This marks data as 'already used for training'.

    Always runs regardless of whether the model improved.
    If skipped, training_data/ is never cleared and the Cloud Scheduler will
    keep finding 1000+ samples and re-triggering the same training indefinitely.

    GCS doesn't have a 'move' operation, so we copy then delete.
    """
    if training_timestamp is None:
        training_timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    archive_prefix = f"trained_data/{training_timestamp}"

    print(f"\n{'='*60}")
    print("ARCHIVING TRAINED DATA")
    print(f"{'='*60}")
    print(f"Moving {len(image_ids)} samples to: {archive_prefix}/")
    print(f"{'='*60}\n")

    moved = 0
    errors = 0

    for image_id in image_ids:
        try:
            # Source paths
            src_image = f"training_data/images/{image_id}.jpg"
            src_label = f"training_data/labels/{image_id}.txt"

            # Destination paths
            dst_image = f"{archive_prefix}/images/{image_id}.jpg"
            dst_label = f"{archive_prefix}/labels/{image_id}.txt"

            # Copy image to archive
            src_blob = bucket.blob(src_image)
            if src_blob.exists():
                bucket.copy_blob(src_blob, bucket, dst_image)
                src_blob.delete()

            # Copy label to archive
            src_label_blob = bucket.blob(src_label)
            if src_label_blob.exists():
                bucket.copy_blob(src_label_blob, bucket, dst_label)
                src_label_blob.delete()

            moved += 1

            if moved % 50 == 0:
                print(f"Archived {moved}/{len(image_ids)} samples...")

        except Exception as e:
            print(f"Error archiving {image_id}: {e}")
            errors += 1

    print(f"\nArchive complete!")
    print(f"  Moved: {moved}")
    print(f"  Errors: {errors}")
    print(f"  Location: gs://{BUCKET_NAME}/{archive_prefix}/")

    return {
        "archive_path": f"gs://{BUCKET_NAME}/{archive_prefix}/",
        "moved": moved,
        "errors": errors,
        "timestamp": training_timestamp
    }

# IMPORTANT: Always keep True for automated runs.
# Setting this to False means training_data/ is never cleared, causing the Cloud Scheduler
# to find 1000+ samples every 3 days and re-trigger training on the same data indefinitely.
ARCHIVE_AFTER_TRAINING = True

if ARCHIVE_AFTER_TRAINING and len(downloaded_image_ids) > 0:
    archive_result = archive_trained_data(downloaded_image_ids)

    # Update summary with archive info
    summary["archive_path"] = archive_result["archive_path"]
    summary["samples_archived"] = archive_result["moved"]

    # Save updated summary
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
else:
    print("Skipping archive (ARCHIVE_AFTER_TRAINING=False or no files to archive)")